<a href="https://colab.research.google.com/github/stefanogiagu/corso_AI_2026/blob/main/ReadGraph_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 706.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 5.8 MB/s eta 0:00:00


In [2]:
from pathlib import Path
from typing import Optional, Tuple, List, Dict, Union
import urllib.request

import torch
from torch_geometric.data import Data


def read_dimacs_col(
    path: Union[str, Path],
    make_undirected: bool = True,
    remove_self_loops: bool = True,
    dtype: torch.dtype = torch.float32,
) -> Data:
    """
    Read a graph coloring instance in DIMACS .col format and convert it
    to a PyTorch Geometric Data object.

    Expected DIMACS format:
        c comment lines
        p edge <num_nodes> <num_edges>
        e <u> <v>

    DIMACS nodes are usually 1-indexed.
    PyTorch Geometric expects 0-indexed node indices.

    Parameters
    ----------
    path:
        Path to the .col file.
    make_undirected:
        If True, each edge (u, v) is stored as both (u, v) and (v, u)
        in edge_index, as commonly expected by PyG message passing layers.
    remove_self_loops:
        If True, discard edges of the form (u, u).
    dtype:
        dtype for the default node features.

    Returns
    -------
    data:
        torch_geometric.data.Data object with fields:
            x: dummy node features, shape [num_nodes, 1]
            edge_index: graph connectivity, shape [2, num_edges_pyg]
            num_nodes: number of nodes
            name: file stem
            num_edges_dimacs: number of undirected edges declared/read
    """

    path = Path(path)

    if path.suffix == ".b" or path.name.endswith(".col.b"):
        raise ValueError(
            f"{path.name} appears to be a binary .col.b file. "
            "This parser only supports standard text DIMACS .col files."
        )

    num_nodes: Optional[int] = None
    declared_num_edges: Optional[int] = None
    edges: List[Tuple[int, int]] = []

    with path.open("r", encoding="utf-8", errors="ignore") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            parts = line.split()

            # Comment line
            if parts[0] == "c":
                continue

            # Problem line: p edge n m
            if parts[0] == "p":
                if len(parts) < 4:
                    raise ValueError(
                        f"Malformed problem line at line {line_number}: {line}"
                    )

                problem_type = parts[1]
                if problem_type not in {"edge", "col"}:
                    # Most coloring instances use 'p edge n m'
                    # but we allow other variants with a warning-like behavior.
                    pass

                num_nodes = int(parts[2])
                declared_num_edges = int(parts[3])
                continue

            # Edge line: e u v
            if parts[0] == "e":
                if len(parts) < 3:
                    raise ValueError(
                        f"Malformed edge line at line {line_number}: {line}"
                    )

                # Convert from 1-indexed DIMACS to 0-indexed PyG.
                u = int(parts[1]) - 1
                v = int(parts[2]) - 1

                if remove_self_loops and u == v:
                    continue

                edges.append((u, v))
                continue

            # Ignore unknown line types, but this can be made stricter if desired.
            # raise ValueError(f"Unknown line type at line {line_number}: {line}")

    if num_nodes is None:
        raise ValueError(f"No problem line 'p edge n m' found in {path}")

    if len(edges) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
    else:
        # Remove duplicate undirected edges robustly.
        # DIMACS stores undirected edges; we canonicalize as (min, max).
        unique_edges = set()
        for u, v in edges:
            if u < 0 or v < 0 or u >= num_nodes or v >= num_nodes:
                raise ValueError(
                    f"Edge ({u + 1}, {v + 1}) out of bounds for graph "
                    f"with {num_nodes} nodes in file {path}"
                )

            a, b = min(u, v), max(u, v)
            unique_edges.add((a, b))

        edge_list = sorted(unique_edges)

        if make_undirected:
            pyg_edges = edge_list + [(v, u) for u, v in edge_list]
        else:
            pyg_edges = edge_list

        edge_index = torch.tensor(pyg_edges, dtype=torch.long).t().contiguous()

    # Dummy node features.
    # For graph coloring, the learnable node embeddings can be created later
    # inside the model, but PyG Data objects usually contain x.
    x = torch.ones((num_nodes, 1), dtype=dtype)

    data = Data(
        x=x,
        edge_index=edge_index,
        num_nodes=num_nodes,
    )

    data.name = path.stem.replace(".col", "")
    data.num_edges_dimacs = len(edges)
    data.num_edges_unique = edge_index.size(1) // 2 if make_undirected else edge_index.size(1)
    data.declared_num_edges = declared_num_edges

    if declared_num_edges is not None and declared_num_edges != len(edges):
        data.edge_count_warning = (
            f"Declared {declared_num_edges} edges, read {len(edges)} edge lines."
        )
    else:
        data.edge_count_warning = None

    return data

In [3]:
data = read_dimacs_col("g22.col")

print(data)
print("Name:", data.name)
print("Number of nodes:", data.num_nodes)
print("DIMACS edges:", data.num_edges_unique)
print("PyG edge_index shape:", data.edge_index.shape)

Data(x=[128, 1], edge_index=[2, 6432], num_nodes=128, name='g22', num_edges_dimacs=6432, num_edges_unique=3216, declared_num_edges=6432)
Name: g22
Number of nodes: 128
DIMACS edges: 3216
PyG edge_index shape: torch.Size([2, 6432])
